In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# RAG 절차
- https://law.go.kr/법령/소득세법 에서 doc다운로드(hwp는 파이썬 못 읽음. pdf는 한글의 경우 짤림) 받아 파일형식을 docx로 변경

1. 문서를 읽는다 (python-docx 이용)
2. 읽어온 문서를 쪼갠다(tiktoken 이용)
    - 모델의 context window를 초과(128,000 context window)
    - 문서가 길면(input이 길면), 비용과 시간이 오래 걸림
3. 쪼갠 문서를 임베딩 -> vector database에 저장 -> chroma(local vector DB), pinecorn(클라우드vector DB)
4. 질문과 vector Database의 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달하여 답변 생성

# 1. 문서를 읽는다 (python-docx 이용)
- pip install python-docx

In [ ]:
%pip install -q python-docx

In [ ]:
from docx import Document
document = Document('./tax_docs/소득세법(법률)(제20615호)(20250701).docx')
print(document)
print(dir(document)) # 객체가 가지고 있는 속성이름

In [ ]:
len(document.paragraphs)
for paragraph in document.paragraphs[:5]:
    #print(dir(paragraph))
    print("paragraph =>", paragraph.text)

In [ ]:
# 문서 전체 내용을 full_text
full_text = ''
for paragraph in document.paragraphs:
    full_text += f"{paragraph.text} "
full_text

In [ ]:
# docx 문서의 글자 및 paragraph 수
len(full_text), len(document.paragraphs)

# 2. 문서를 쪼갠다
- pip install tiktoken

- full_text -> 토큰단위로 쪼개서 숫자
- 1500토큰씩 문서를 쪼개기

In [ ]:
%pip install -q tiktoken

In [ ]:
import tiktoken
# tiktoken이 인식 가능한 모델 이름만 사용 가능 : gpt-4X, gpt-3.5-turbo, text-embedding-ada-002,.
# gpt-1.4-nano는 불가
encoder = tiktoken.encoding_for_model("gpt-4o-mini")
# 문자들 -> 숫자 리스트
encoding = encoder.encode(full_text)
# 숫자리스트 -> 문자들
decoded = encoder.decode(encoding)

In [ ]:
print('full_text의 전체 토큰수 :',len(encoding))
print('decoded :', decoded[:10])
print('encoding :', encoding[:10])

In [ ]:
encoder.encode("소득세"), encoder.encode(" 친구"), encoder.encode("Hello"), encoder.encode("안녕")

In [ ]:
list(range(0, 15300, 1500))

In [ ]:
# full_text를 쪼개는 함수 : chunk_list 반환
import tiktoken
def split_text(full_text, chunk_size):
    encoder = tiktoken.encoding_for_model("gpt-4o-mini")
    total_encoding = encoder.encode(full_text)
    total_token_count = len(total_encoding)
    chunk_list = []
    for i in range(0, total_token_count, chunk_size):
        chunk = total_encoding[i : i+chunk_size]
        decoded = encoder.decode(chunk)
        chunk_list.append(decoded)
    return chunk_list

In [ ]:
example_text = "소득세법 법률 일부개정함 이자소득 배당소득 안녕 홍길동"
encoder = tiktoken.encoding_for_model("gpt-4o-mini")
encoding = encoder.encode(example_text)
len(encoding)

In [ ]:
split_text(example_text, 10)

In [ ]:
chunk_list = split_text(full_text, 1500)

In [ ]:
len(chunk_list)

# 3. 쪼갠 문서를 임베딩 -> vector database에 저장
- chroma 공식 홈페이지(langchain chroma)
- pip install chromadb

In [ ]:
%pip install -q chromadb

In [ ]:
import chromadb
chroma_client = chromadb.Client()

In [ ]:
# collection 은 RDB의 테이블과 같은 개념
collection_name = "tax_collection"


In [ ]:
# 임베딩 객체
from dotenv import load_dotenv
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

load_dotenv()
import os
openai_api_key = os.getenv("OPENAI_API_KEY")

openai_embedding = OpenAIEmbeddingFunction(
    api_key=openai_api_key,
    model_name="text-embedding-3-large"
)

In [ ]:
# chroma db에 collection 생성
tax_collection = chroma_client.create_collection(
    name=collection_name,
    embedding_function=openai_embedding
)

In [ ]:
# collction에 입력할 때 사용할 id들 : 문자
ids = [str(id) for id in range(len(chunk_list))]
print(ids[:5])

In [ ]:
tax_collection.add(documents=chunk_list,
                  ids=ids)

# 4. 질문과 vector Database의 유사도 검색

In [ ]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = tax_collection.query(
    query_texts=[query],
    n_results=3,
)

In [ ]:
retrieved_docs['documents'][0][2]

# 5. 유사도 검색으로 나온 문서를 LLM에 질문과 같이 전달
- retrieved_doc['documents'][0]

In [ ]:
from openai import OpenAI
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4o-mini", # tiktoken.encoding_for_model("gpt-4o-mini")
    messages=[
        {"role":"system",
         "content": f"""당신은 한국 소득세 법에 대한 전문가입니다. 아래의 내용을 참고해서
         질문에 답변하세요. 만약 답변할 수 없다면 '모르겠습니다.'라고 답변하세요.
         {retrieved_docs['documents'][0]}
         """
        },
        {"role":"user", "content": query}
    ]
)

In [ ]:
# 답변 출력
print(response.choices[0].message.content)